# RCDM Training on Google Colab

This notebook trains the RCDM (Representation-Conditioned Diffusion Model) on Tiny ImageNet using Google Colab's free GPU.

## Setup Steps:
1. **Runtime → Change runtime type → GPU (T4 recommended)**
2. Run cells top to bottom
3. When prompted, sign in to Google Drive (for saving checkpoints)
4. Upload `train_reps.pt` to Google Drive beforehand for fastest data access

## Repo:
[github.com/SeverinLe/master_implementation](https://github.com/SeverinLe/master_implementation)

## 1. Check GPU Availability

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB moin")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 101.97 GB moin


## 2. Mount Google Drive (Optional - for storing checkpoints)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
!mkdir -p /content/drive/MyDrive/rcdm_checkpoints

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Clone Code from GitHub

In [3]:
import os
import shutil

REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
REPO_DIR = "/content/master_implementation"
OLD_DIR  = "/master_implementation"   # path used by earlier sessions

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo found at correct location, pulling latest changes...")
    !git -C {REPO_DIR} pull

elif os.path.isdir(os.path.join(OLD_DIR, ".git")):
    # Repo was cloned to the old root-level path — migrate it
    print(f"Migrating repo from {OLD_DIR} → {REPO_DIR} ...")
    # Preserve data/ subfolder if it already exists in the destination
    data_backup = None
    if os.path.isdir(os.path.join(REPO_DIR, "data")):
        data_backup = "/tmp/_rcdm_data_backup"
        shutil.copytree(os.path.join(REPO_DIR, "data"), data_backup, dirs_exist_ok=True)
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    shutil.copytree(OLD_DIR, REPO_DIR)
    if data_backup:
        shutil.copytree(data_backup, os.path.join(REPO_DIR, "data"), dirs_exist_ok=True)
    !git -C {REPO_DIR} pull

else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

print("\nProject structure:")
for item in sorted(os.listdir(REPO_DIR)):
    print(f"  {item}")

Repo already exists, pulling latest changes...
fatal: not a git repository (or any of the parent directories): .git
/content/master_implementation

Project structure:
  data


> **Tip:** If you push new changes from Cursor, re-run the cell above to `git pull` them into Colab without restarting the runtime.

In [4]:
# Verify the clone contains all expected modules
!ls rcdm/ scripts/ guided_diffusion/guided_diffusion/

ls: cannot access 'rcdm/': No such file or directory
ls: cannot access 'scripts/': No such file or directory
ls: cannot access 'guided_diffusion/guided_diffusion/': No such file or directory


## 4. Install Dependencies

In [5]:
# Install guided_diffusion in editable mode (absolute path avoids cwd issues)
!pip install -e /content/master_implementation/guided_diffusion/ -q

# Other dependencies
!pip install "blobfile>=1.0.5" tqdm pillow torchvision -q

print("✓ Dependencies installed")

ERROR: guided_diffusion/ is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
✓ Dependencies installed


## 5. Get Training Data

`train_reps.pt` (~826MB) is not stored in GitHub (too large). Two options:

- **Option A (recommended):** Upload it to Google Drive once, then copy with the cell below — fast and persists across sessions.
- **Option B:** Upload directly to Colab — works but takes longer and is lost when the session ends.

In [6]:
# Mount Drive and copy train_reps.pt to the expected location

import os
import shutil
import torch
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = "/content/master_implementation"
DATA_DIR = os.path.join(REPO_DIR, "data", "tiny-imagenet-200")
os.makedirs(DATA_DIR, exist_ok=True)

SRC_PATH = "/content/drive/MyDrive/master_implementation/train_reps.pt"
DST_PATH = os.path.join(DATA_DIR, "train_reps.pt")

assert os.path.exists(SRC_PATH), f"Source file not found: {SRC_PATH}"

shutil.copy2(SRC_PATH, DST_PATH)

obj = torch.load(DST_PATH, map_location="cpu")
print("Loaded OK:", DST_PATH)
print("Type:", type(obj))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded OK: /content/master_implementation/data/tiny-imagenet-200/train_reps.pt
Type: <class 'dict'>


In [7]:
# To save train_reps.pt to your Drive for future sessions (run once after uploading):
# !cp data/tiny-imagenet-200/train_reps.pt /content/drive/MyDrive/train_reps.pt

## 6. Verify Setup

In [8]:
import os
import sys

REPO_DIR = "/content/master_implementation"

# repo root → needed for rcdm
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# guided_diffusion parent → needed so unet.py can reach rcdm (both live under REPO_DIR)
GD_PARENT = os.path.join(REPO_DIR, "guided_diffusion")
if GD_PARENT not in sys.path:
    sys.path.insert(0, GD_PARENT)

# Test imports
from guided_diffusion.script_util import create_model_and_diffusion
from rcdm.dataset import RepresentationDataset
from rcdm.encoder import load_encoder

print(f"guided_diffusion: {create_model_and_diffusion.__name__}")
print(f"rcdm.dataset: {RepresentationDataset.__name__}")
print(f"rcdm.encoder: {load_encoder.__name__}")

# Check data file exists at the path expected by scripts/train.py
data_path = os.path.join(REPO_DIR, "data", "tiny-imagenet-200", "train_reps.pt")
assert os.path.exists(data_path), f"Data file not found: {data_path}"

print(f"\nAll imports OK")
print(f"Data file found: {os.path.getsize(data_path) / 1e9:.2f} GB")

ModuleNotFoundError: No module named 'rcdm'

In [ ]:
# (cell removed — verification is handled by Cell 15 above)


## 7. Start Training

Run your training script with GPU acceleration!

In [ ]:
# Training with GPU
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --device cuda

## 8. Monitor Training (Optional - Run in separate cell)

In [ ]:
# Check latest checkpoint
!ls -lh /content/drive/MyDrive/rcdm_checkpoints/

# Monitor GPU usage
!nvidia-smi

## 9. Generate Samples (After Training)

In [ ]:
# Load your sampling script here once you create it
# !python scripts/sampling.py --checkpoint /content/drive/MyDrive/rcdm_checkpoints/model_100000.pt

---

## Tips

| Topic | Guidance |
|-------|----------|
| **GPU runtime limit** | Free Colab = ~12 h. Save checkpoints frequently (`--save_interval 2000`). |
| **Batch size** | T4: use 64–128. A100/H100: up to 256. |
| **Checkpoints** | Saved to Google Drive — survive session restarts. |
| **Code changes** | Edit in Cursor → `git push` → re-run Cell 3 (`git pull`) in Colab. No restart needed. |
| **Monitor GPU** | Run `!nvidia-smi` in a spare cell at any time. |

## Resume after disconnection

Re-run cells 1–6 (clone/pull, install, data), then:

```python
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --resume /content/drive/MyDrive/rcdm_checkpoints/rcdm_stepXXXXXXX.pt \
    --device cuda
```